# PCA for Neuropixels Reach Task (Block Design)

This notebook replaces the older `PCA_nerual_data_2_*.ipynb` workflow with a reproducible pipeline.
It is built for alternating reach blocks:

- `baseline` (no opto)
- `opto_epoch_k`
- `washout_epoch_k`

in fixed or configurable trial lengths (default 20 trials/block).


## What This Fixes vs Legacy Notebooks

- Removes hard-coded absolute paths.
- Removes undefined-variable dependency (`Xr` before definition).
- Prevents out-of-bounds and mismatched trial-label indexing.
- Adds explicit shape checks and trial-type validation.
- Keeps PCA behavior consistent with prior approach: z-score per neuron, then PCA over time/trials.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

sns.set_context("talk")
sns.set_style("white")


In [ ]:
# -----------------------------
# USER CONFIG
# -----------------------------

ROOT = Path.cwd()

# Expected data format:
# A) (n_units, n_trials, n_bins) -> auto-transposed
# B) (n_trials, n_units, n_bins) -> used as-is
DATA_PATH = ROOT / "extra_files" / "reach1_4_1_10_window_50ms_FR_per_trial_B.npy"

GOOD_UNITS = None
BIN_SIZE_S = 0.05
FRAMES_PRE = 10
FRAMES_POST = 180
N_COMPONENTS = 12
SMOOTH_SIGMA = 2

BASELINE_TRIALS = 20
TRIALS_PER_BLOCK = 20
N_ALTERNATING_BLOCKS = 4

# Optional exact blocks: [(label, start_inclusive, end_exclusive), ...]
CUSTOM_BLOCKS = None


In [ ]:
def ensure_trials_shape(trials: np.ndarray) -> np.ndarray:
    """Return array shaped (n_trials, n_units, n_bins)."""
    if trials.ndim != 3:
        raise ValueError(f"Expected 3D array, got shape {trials.shape}")
    if trials.shape[0] < trials.shape[1]:
        trials = np.transpose(trials, (1, 0, 2))
    return trials


def build_blocks(n_trials: int):
    if CUSTOM_BLOCKS is not None:
        blocks = CUSTOM_BLOCKS
    else:
        blocks = [("baseline", 0, BASELINE_TRIALS)]
        cursor = BASELINE_TRIALS
        epoch = 1
        for i in range(N_ALTERNATING_BLOCKS):
            label = f"opto_epoch_{epoch}" if i % 2 == 0 else f"washout_epoch_{epoch}"
            blocks.append((label, cursor, min(cursor + TRIALS_PER_BLOCK, n_trials)))
            cursor += TRIALS_PER_BLOCK
            if i % 2 == 1:
                epoch += 1

    clean = []
    for label, s, e in blocks:
        s = max(0, int(s))
        e = min(int(e), n_trials)
        if e > s:
            clean.append((label, s, e))

    if not clean:
        raise ValueError("No valid blocks after validation.")
    return clean


def zscore_rows(x: np.ndarray) -> np.ndarray:
    scaler = StandardScaler(with_mean=True, with_std=True)
    return scaler.fit_transform(x.T).T


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Data file not found: {DATA_PATH}")

raw = np.load(DATA_PATH, allow_pickle=True)
trials = ensure_trials_shape(np.asarray(raw))

if GOOD_UNITS is not None:
    trials = trials[:, GOOD_UNITS, :]

n_trials, n_units, n_bins = trials.shape
print(f"Loaded trials shape: {trials.shape} (trials, units, bins)")

if not (0 <= FRAMES_PRE < FRAMES_POST <= n_bins):
    raise ValueError(
        f"Invalid reach window: FRAMES_PRE={FRAMES_PRE}, FRAMES_POST={FRAMES_POST}, n_bins={n_bins}"
    )

time = np.arange(n_bins) * BIN_SIZE_S


In [ ]:
blocks = build_blocks(n_trials)

trial_type = np.array(["unlabeled"] * n_trials, dtype=object)
for label, s, e in blocks:
    trial_type[s:e] = label

valid_mask = trial_type != "unlabeled"
if valid_mask.sum() == 0:
    raise ValueError("No labeled trials. Check block definitions.")

trials_labeled = trials[valid_mask]
trial_type_labeled = trial_type[valid_mask]
trial_types = np.array(pd.unique(trial_type_labeled), dtype=object)
t_type_ind = [np.where(trial_type_labeled == t)[0] for t in trial_types]

print("Blocks:")
for b in blocks:
    print("  ", b)
print("\nLabeled trial count:", len(trials_labeled))
print("Trial types:", list(trial_types))


In [ ]:
X_trial = np.vstack([
    t[:, FRAMES_PRE:FRAMES_POST].mean(axis=1) for t in trials_labeled
]).T

X_trial_z = zscore_rows(X_trial)
pca_trial = PCA(n_components=min(N_COMPONENTS, X_trial_z.shape[0], X_trial_z.shape[1]))
Xp = pca_trial.fit_transform(X_trial_z.T).T

print("Explained variance ratio (first 5):", np.round(pca_trial.explained_variance_ratio_[:5], 4))


In [ ]:
projections = [(0, 1), (1, 2), (0, 2)]
pal = sns.color_palette("colorblind", len(trial_types))

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharex=False, sharey=False)
for ax, (i, j) in zip(axes, projections):
    for k, t in enumerate(trial_types):
        idx = t_type_ind[k]
        if len(idx) == 0:
            continue
        ax.scatter(Xp[i, idx], Xp[j, idx], s=35, alpha=0.85, color=pal[k], label=str(t))
    ax.set_xlabel(f"PC {i+1}")
    ax.set_ylabel(f"PC {j+1}")

axes[-1].legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
sns.despine()
plt.tight_layout()
plt.show()


In [ ]:
trial_averages = []
for idx in t_type_ind:
    if len(idx) == 0:
        continue
    trial_averages.append(trials_labeled[idx].mean(axis=0))

if len(trial_averages) < 2:
    raise ValueError("Need at least two non-empty trial types for trajectory comparison.")

Xa = np.hstack(trial_averages)
Xa_z = zscore_rows(Xa)

pca_traj = PCA(n_components=min(N_COMPONENTS, Xa_z.shape[0], Xa_z.shape[1]))
Xa_p = pca_traj.fit_transform(Xa_z.T).T

print("Trajectory PCA explained variance (first 5):", np.round(pca_traj.explained_variance_ratio_[:5], 4))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharex=True)
for comp in range(min(3, Xa_p.shape[0])):
    ax = axes[comp]
    for k, t in enumerate(trial_types):
        s = k * n_bins
        e = (k + 1) * n_bins
        x = Xa_p[comp, s:e]
        if SMOOTH_SIGMA and SMOOTH_SIGMA > 0:
            from scipy.ndimage import gaussian_filter1d
            x = gaussian_filter1d(x, sigma=SMOOTH_SIGMA)
        ax.plot(time, x, color=pal[k], lw=2, label=str(t))

    ax.axvline(time[FRAMES_PRE], color="gray", ls="--", lw=1)
    ax.axvline(time[FRAMES_POST - 1], color="gray", ls="--", lw=1)
    ax.set_ylabel(f"PC {comp+1}")

axes[1].set_xlabel("Time (s)")
axes[-1].legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
sns.despine()
plt.tight_layout()
plt.show()


In [ ]:
OUT_DIR = ROOT / "extra_files"
OUT_DIR.mkdir(parents=True, exist_ok=True)

np.save(OUT_DIR / f"{DATA_PATH.stem}_trial_pca_scores.npy", Xp)
np.save(OUT_DIR / f"{DATA_PATH.stem}_trajectory_pca_scores.npy", Xa_p)
np.save(OUT_DIR / f"{DATA_PATH.stem}_trial_labels.npy", trial_type_labeled)

print("Saved:")
print(" ", OUT_DIR / f"{DATA_PATH.stem}_trial_pca_scores.npy")
print(" ", OUT_DIR / f"{DATA_PATH.stem}_trajectory_pca_scores.npy")
print(" ", OUT_DIR / f"{DATA_PATH.stem}_trial_labels.npy")
